# RAG-система: юридические документы (RU) и техподдержка IBM (EN)

**Retrieval-Augmented Generation**: по вопросу находим релевантные фрагменты документов и передаём их LLM, которая отвечает только на их основе.

Пайплайн:
1. **Чанкинг** — документы режутся на фрагменты ~1000 символов с перекрытием; к каждому чанку дописываем название документа (и номер статьи для законов).
2. **Retrieval** — BM25 (по стеммам), dense bi-encoder (E5), гибрид через Reciprocal Rank Fusion.
3. **Reranking** — cross-encoder пересортировывает top-50 кандидатов.
4. **Generation** — `Qwen/Qwen2.5-1.5B-Instruct` отвечает по top-4 чанкам, ссылается на источники, отказывается отвечать, если ответа в контексте нет.

Данные:
- `parlorsky/legal-rag-benchmark-ru` — 12 нормативных документов (законы, положения ЦБ) и 25 вопросов с эталонными ответами;
- `nvidia/TechQA-RAG-Eval` — 28 тыс. технотов IBM и 910 вопросов (300 из них без ответа в корпусе).

Оценка: качество поиска (Hit@k, MRR, доля слов эталонного ответа в найденных чанках) и качество ответа (token F1, семантическая близость к эталону, доля отказов) в сравнении с LLM без RAG.

In [2]:
import os, re, json, zipfile, random
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from tqdm.auto import tqdm
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfFileSystem, hf_hub_download

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CACHE = 'rag_cache'
os.makedirs(CACHE, exist_ok=True)
random.seed(42)
np.random.seed(42)
pd.set_option('display.max_colwidth', 200)
device

'cuda'

## Общие компоненты
### Чанкинг
Склеиваем абзацы, пока чанк не превысит `size` символов; следующий чанк начинается с хвоста предыдущего (`overlap` символов), чтобы не терять мысль на границе. Если задан `header_re` (например, `Статья N`), заголовок текущего раздела дописывается в начало каждого его чанка — так чанк из середины статьи остаётся привязан к её номеру и названию.

In [3]:
def chunk_text(text, size=1000, overlap=200, header_re=None):
    pieces = []
    for p in (p.strip() for p in text.split('\n')):
        while len(p) > size:  # слишком длинный абзац режем по пробелу
            cut = p.rfind(' ', 0, size)
            cut = cut if cut > size // 2 else size
            pieces.append(p[:cut])
            p = p[cut:].strip()
        if p:
            pieces.append(p)

    chunks, cur, header = [], '', ''
    for p in pieces:
        is_header = bool(header_re and re.match(header_re, p))
        if cur and len(cur) + len(p) > size:
            chunks.append(cur)
            tail = '' if is_header else cur[-overlap:].split(' ', 1)[-1] if overlap else ''
            cur = f'{header}\n{tail}' if header and not is_header else tail
        if is_header:
            header = p[:200]
        cur = f'{cur}\n{p}'.strip()
    if cur:
        chunks.append(cur)
    return chunks


def build_chunks(docs, size, overlap, header_re=None, title_fn=lambda name, text: name):
    rows = []
    for name, text in tqdm(docs.items(), desc='chunking'):
        title = title_fn(name, text)
        rows += [{'doc': name, 'text': f'{title}\n{c}'} for c in chunk_text(text, size, overlap, header_re)]
    return pd.DataFrame(rows)

### BM25
Свой BM25 на разреженных матрицах: веса `idf * tf*(k1+1) / (tf + k1*(1-b+b*len/avg_len))` считаются один раз при индексации, поиск — одно умножение матрицы на вектор запроса. Слова приводим к стеммам (Snowball), чтобы «кредитной» и «кредитная» совпадали.

In [4]:
STEMMERS = {'ru': SnowballStemmer('russian'), 'en': SnowballStemmer('english')}

def make_tokenizer(lang):
    stem = lru_cache(maxsize=None)(STEMMERS[lang].stem)
    return lambda text: [stem(w) for w in re.findall(r'[^\W_]+', text.lower())]


class BM25:
    def __init__(self, texts, tokenize, k1=1.5, b=0.75):
        self.cv = CountVectorizer(analyzer=tokenize)
        tf = self.cv.fit_transform(texts).astype(np.float32).tocoo()
        dl = np.bincount(tf.row, weights=tf.data, minlength=tf.shape[0])
        df = np.bincount(tf.col, minlength=tf.shape[1])
        idf = np.log(1 + (tf.shape[0] - df + 0.5) / (df + 0.5))
        w = tf.data * (k1 + 1) / (tf.data + k1 * (1 - b + b * dl[tf.row] / dl.mean())) * idf[tf.col]
        self.W = sp.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape)

    def scores(self, q):
        v = self.cv.transform([q])
        v.data[:] = 1
        return (self.W @ v.T).toarray().ravel()

### Индекс: BM25 + dense + гибрид + реранкер
- `bm25` / `dense` — top-`n_cand` по одному методу;
- `hybrid` — объединение двух списков через **Reciprocal Rank Fusion**: `score = Σ 1/(60 + rank)`; не нужно калибровать шкалы BM25 и косинуса между собой;
- `rerank=True` — кандидаты пересортировываются cross-encoder'ом (видит запрос и чанк одновременно, точнее, но медленнее).

Эмбеддинги чанков кэшируются в `rag_cache/`.

In [5]:
def top(scores, k):
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(-scores[idx])]


def rrf(rankings, c=60):
    s = Counter()
    for r in rankings:
        for rank, i in enumerate(r):
            s[i] += 1 / (c + rank + 1)
    return np.array([i for i, _ in s.most_common()])


class Index:
    def __init__(self, chunks, lang, embedder, reranker, q_prefix='query: ', p_prefix='passage: ', cache=None):
        self.chunks = chunks.reset_index(drop=True)
        self.texts = self.chunks['text'].tolist()
        self.docs = self.chunks['doc'].values
        self.embedder, self.reranker, self.q_prefix = embedder, reranker, q_prefix
        self.bm25 = BM25(self.texts, make_tokenizer(lang))
        if cache and os.path.exists(cache):
            self.emb = np.load(cache)
        else:
            self.emb = embedder.encode([p_prefix + t for t in self.texts], batch_size=64, normalize_embeddings=True,
                                       show_progress_bar=True, convert_to_numpy=True)
            if cache:
                np.save(cache, self.emb)
        assert len(self.emb) == len(self.texts), 'кэш эмбеддингов от другого чанкинга — удалите файл'

    def dense_scores(self, q):
        return self.emb @ self.embedder.encode(self.q_prefix + q, normalize_embeddings=True)

    def search(self, q, k=5, mode='hybrid', rerank=False, n_cand=50):
        if mode == 'bm25':
            idx = top(self.bm25.scores(q), n_cand)
        elif mode == 'dense':
            idx = top(self.dense_scores(q), n_cand)
        else:
            idx = rrf([top(self.bm25.scores(q), n_cand), top(self.dense_scores(q), n_cand)])[:n_cand]
        if rerank:
            s = self.reranker.predict([(q, self.texts[i]) for i in idx], batch_size=32, show_progress_bar=False)
            idx = idx[np.argsort(-s)]
        return idx[:k]

### Генератор: Qwen2.5-1.5B-Instruct
Промпт: отвечать только по фрагментам, ссылаться на номера фрагментов, при отсутствии ответа — фиксированная фраза отказа (по ней считаем долю отказов). Декодирование жадное, чтобы результаты были воспроизводимы.

In [6]:
LLM = 'Qwen/Qwen2.5-1.5B-Instruct'
llm_tok = AutoTokenizer.from_pretrained(LLM)
dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device == 'cuda' else torch.float32
llm = AutoModelForCausalLM.from_pretrained(LLM, torch_dtype=dtype).to(device).eval()


llm_tok.padding_side = 'left'  # для батчевой генерации


@torch.no_grad()
def generate_batch(messages_list, max_new_tokens=200, bs=8):
    texts = [llm_tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in messages_list]
    order = np.argsort([len(t) for t in texts])  # похожие по длине вместе — меньше паддинга
    res = [None] * len(texts)
    for s in tqdm(range(0, len(texts), bs), desc='generate'):
        ids = order[s:s + bs]
        enc = llm_tok([texts[i] for i in ids], return_tensors='pt', padding=True).to(device)
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
                           top_k=None, repetition_penalty=1.05, pad_token_id=llm_tok.eos_token_id)
        for i, o in zip(ids, out):
            res[i] = llm_tok.decode(o[enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return res


def generate(messages, max_new_tokens=300):
    return generate_batch([messages], max_new_tokens, bs=1)[0]


PROMPTS = {
    'ru': {
        'refusal': 'В документах нет ответа',
        'system': ('Ты — юридический ассистент. Отвечай на вопрос только на основе приведённых фрагментов '
                   'нормативных документов, близко к формулировкам закона. После каждого утверждения указывай номер '
                   'фрагмента в квадратных скобках, например [1]. Если во фрагментах нет ответа, ответь ровно: '
                   '«В документах нет ответа».'),
        'user': 'Фрагменты документов:\n\n{ctx}\n\nВопрос: {q}\nОтвет:',
        'plain': 'Ты — юридический ассистент. Кратко и точно ответь на вопрос по законодательству РФ.',
    },
    'en': {
        'refusal': "I don't know",
        'system': ('You are an IBM technical support assistant. Answer the question using only the provided documents. '
                   'Be concise and specific: give the fix, command or explanation from the documents and cite them as [1], [2]. '
                   "If the documents do not contain the answer, reply exactly: I don't know."),
        'user': 'Documents:\n\n{ctx}\n\nQuestion: {q}\nAnswer:',
        'plain': 'You are an IBM technical support assistant. Answer the question concisely.',
    },
}


def rag_messages(index, q, lang, k=4, mode='hybrid', rerank=True):
    idx = index.search(q, k=k, mode=mode, rerank=rerank)
    ctx = '\n\n'.join(f'[{j + 1}] {index.texts[i]}' for j, i in enumerate(idx))
    p = PROMPTS[lang]
    return [{'role': 'system', 'content': p['system']},
            {'role': 'user', 'content': p['user'].format(ctx=ctx, q=q)}], idx


def rag_answer(index, q, lang, **kw):
    msgs, idx = rag_messages(index, q, lang, **kw)
    return generate(msgs), idx


def plain_answer(q, lang):
    return generate([{'role': 'system', 'content': PROMPTS[lang]['plain']}, {'role': 'user', 'content': q}])


def ask(index, q, lang, **kw):
    ans, idx = rag_answer(index, q, lang, **kw)
    print(f'Вопрос: {q}\n\nОтвет: {ans}\n\nИсточники:')
    for j, i in enumerate(idx):
        print(f'  [{j + 1}] {index.docs[i][:70]} | {index.texts[i].split(chr(10), 1)[-1][:150]!r}')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

c:\Users\Lenovo\miniconda3\envs\jupyter_env\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\Lenovo\miniconda3\envs\jupyter_env\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

### Метрики
**Retrieval** (на каждом вопросе берём top-10 чанков):
- `Hit@k` — среди top-k есть чанк из нужного документа; `MRR@10` — 1/позиция первого такого чанка;
- `AnsRecall@5` — максимальная по top-5 чанкам доля стеммов эталонного ответа, встречающихся в чанке (насколько найден именно фрагмент с ответом, а не просто нужный документ).

**Generation**:
- `F1` — token F1 по стеммам между ответом и эталоном (ссылки `[1]` убираем);
- `SemSim` — косинус E5-эмбеддингов ответа и эталона;
- `Refusal` — доля ответов-отказов.

In [7]:
def token_f1(pred, gold, tokenize):
    p, g = tokenize(pred), tokenize(gold)
    n = sum((Counter(p) & Counter(g)).values())
    if n == 0:
        return 0.0
    pr, rc = n / len(p), n / len(g)
    return 2 * pr * rc / (pr + rc)


def answer_recall(text, gold, tokenize):
    g = set(tokenize(gold))
    return len(g & set(tokenize(text))) / max(len(g), 1)


RETRIEVERS = {
    'BM25':             dict(mode='bm25', rerank=False),
    'Dense':            dict(mode='dense', rerank=False),
    'Hybrid (RRF)':     dict(mode='hybrid', rerank=False),
    'Dense + rerank':   dict(mode='dense', rerank=True),
    'Hybrid + rerank':  dict(mode='hybrid', rerank=True),
}


def eval_retrieval(index, qa, lang, ks=(1, 3, 5, 10)):
    tokenize = make_tokenizer(lang)
    rows = {}
    for name, cfg in RETRIEVERS.items():
        hits, rr, ar = {k: [] for k in ks}, [], []
        for q, gold_docs, gold in tqdm(zip(qa['question'], qa['gold_docs'], qa['answer']), total=len(qa), desc=name):
            idx = index.search(q, k=max(ks), **cfg)
            pos = next((j for j, i in enumerate(idx) if index.docs[i] in gold_docs), None)
            for k in ks:
                hits[k].append(pos is not None and pos < k)
            rr.append(0 if pos is None else 1 / (pos + 1))
            ar.append(max(answer_recall(index.texts[i], gold, tokenize) for i in idx[:5]))
        rows[name] = {**{f'Hit@{k}': np.mean(v) for k, v in hits.items()}, 'MRR@10': np.mean(rr), 'AnsRecall@5': np.mean(ar)}
    return pd.DataFrame(rows).T.round(3)


def run_generation(index, questions, lang, configs):
    # configs: {имя: параметры rag_messages или None для LLM без RAG}; генерация идёт батчами
    out = {}
    for name, cfg in configs.items():
        if cfg is None:
            msgs = [[{'role': 'system', 'content': PROMPTS[lang]['plain']}, {'role': 'user', 'content': q}]
                    for q in questions]
        else:
            msgs = [rag_messages(index, q, lang, **cfg)[0] for q in tqdm(questions, desc=f'{name}: поиск')]
        out[name] = generate_batch(msgs)
    return out


def score_answers(preds, golds, lang, embedder):
    tokenize = make_tokenizer(lang)
    preds = [re.sub(r'\[\d+\]', '', p) for p in preds]
    e = embedder.encode(['query: ' + t for t in preds + list(golds)], normalize_embeddings=True)
    return {
        'F1': np.mean([token_f1(p, g, tokenize) for p, g in zip(preds, golds)]),
        'SemSim': float(np.mean((e[:len(preds)] * e[len(preds):]).sum(1))),
        'Refusal': np.mean([PROMPTS[lang]['refusal'].lower() in p.lower() for p in preds]),
    }


GEN_CONFIGS = {
    'LLM без RAG':           None,
    'RAG: dense':            dict(mode='dense', rerank=False),
    'RAG: hybrid + rerank':  dict(mode='hybrid', rerank=True),
}

## Часть 1. Юридические документы (legal-rag-benchmark-ru)
Часть файлов содержит в имени символы, недопустимые в Windows (`"`), поэтому читаем их через `HfFileSystem` прямо в память и кэшируем одним json.

In [8]:
LEGAL = 'datasets/parlorsky/legal-rag-benchmark-ru'
fs = HfFileSystem()

legal_path = f'{CACHE}/legal_docs.json'
if os.path.exists(legal_path):
    legal_docs = json.load(open(legal_path, encoding='utf-8'))
else:
    legal_docs = {os.path.basename(p)[:-4].strip(' •'): fs.read_text(p, encoding='utf-8', errors='replace')
                  for p in fs.glob(f'{LEGAL}/sample_docs/*.txt')}
    json.dump(legal_docs, open(legal_path, 'w', encoding='utf-8'), ensure_ascii=False)

legal_qa = pd.DataFrame(json.loads(fs.read_text(f'{LEGAL}/test_dataset.json', encoding='utf-8')))
legal_qa = legal_qa.rename(columns={'expected_answer': 'answer'})

pd.Series({t: len(s) for t, s in legal_docs.items()}, name='символов').sort_values()

Федеральный закон № 224-ФЗ «О государственно-частном партнёрстве (ГЧП)» – регулирует финансирование инфраструктурных проектов.        5993
Федеральный закон № 367-ФЗ «О внесении изменений в Закон о залоге» – упрощает залог будущих активов в проектном финансировании.      65863
265_Фонд_микрокредитования_Ирк_обл                                                                                                   87059
Федеральный закон от 31 декабря 2014 г N 488 ФЗ О промышленной политике в Россий                                                    162372
Постановление Правительства РФ от 11 октября 2014 г N 1044 Об утверждении Програ                                                    168973
Положение Банка России № 590-П (06.07.2022)                                                                                         174071
Федеральный закон № 218-ФЗ "О публично-правовой компании «Фонд защиты прав граждан — участников долевого строительства»"            178379
Федеральный закон от 30 дек

In [9]:
legal_qa[['id', 'question_type', 'difficulty', 'question', 'answer']].head()

,id,question_type,difficulty,question,answer
0,def_01,definition,easy,Что такое кредитная организация согласно закону о банках?,"Кредитная организация - юридическое лицо, которое для извлечения прибыли как основной цели своей деятельности на основании специального разрешения (лицензии) Центрального банка Российской Федераци..."
1,def_02,comparison,medium,Чем отличается банк с универсальной лицензией от банка с базовой лицензией?,"Банк с универсальной лицензией имеет право осуществлять банковские операции, указанные в части первой статьи 5 закона о банках. Банк с базовой лицензией имеет право осуществлять те же операции, но..."
2,def_03,definition,easy,Что такое банковская группа?,"Банковской группой признается не являющееся юридическим лицом объединение юридических лиц, в котором одно юридическое лицо или несколько юридических лиц (участники банковской группы) находятся под..."
3,def_04,factual,medium,Какие банковские операции относятся к основным согласно закону о банках?,К банковским операциям относятся: привлечение денежных средств физических и юридических лиц во вклады (до востребования и на определенный срок); размещение указанных привлеченных средств от своего...
4,def_05,definition,medium,Что такое небанковская кредитная организация?,"Небанковская кредитная организация — это кредитная организация, имеющая право осуществлять ограниченный набор банковских операций. Существует несколько видов: организации, имеющие право на осущест..."


Названия в `source_docs` не совпадают с именами файлов один в один (обрезаны, без кавычек) — сопоставляем по доле общих слов.

In [10]:
def words(s):
    return set(re.findall(r'[^\W_]+', s.lower()))

def match_doc(name):
    return max(legal_docs, key=lambda t: len(words(name) & words(t)) / len(words(name)))

legal_qa['gold_docs'] = legal_qa['source_docs'].apply(lambda ds: {match_doc(d) for d in ds})
{d: match_doc(d) for d in sorted({d for ds in legal_qa['source_docs'] for d in ds})}

{'265_Фонд_микрокредитования_Ирк_обл': '265_Фонд_микрокредитования_Ирк_обл',
 'Федеральный закон от 02.12.90 N 395-I О банках и банковской деятельности': 'Федеральный закон от 02.12.90 N 395-I О банках и банковской деятельности',
 'Федеральный закон от 30 декабря 2004 г N 214 ФЗ Об участии в долевом строительст': 'Федеральный закон от 30 декабря 2004 г N 214 ФЗ Об участии в долевом строительст',
 'Федеральный закон от 31 декабря 2014 г N 488 ФЗ О промышленной политике в Россий': 'Федеральный закон от 31 декабря 2014 г N 488 ФЗ О промышленной политике в Россий',
 'Федеральный закон № 224-ФЗ О государственно-частном партнёрстве': 'Федеральный закон № 224-ФЗ «О государственно-частном партнёрстве (ГЧП)» – регулирует финансирование инфраструктурных проектов.'}

In [11]:
legal_chunks = build_chunks(legal_docs, size=1000, overlap=200, header_re=r'Статья\s+\d+')
print(len(legal_chunks))
print(legal_chunks['doc'].value_counts().to_string())
print('\n' + legal_chunks.loc[legal_chunks['doc'].str.contains('395-I'), 'text'].iloc[20])

chunking:   0%|          | 0/12 [00:00<?, ?it/s]

8760
doc
Указание-Банка-России-от-08.10.2018-N-4927-У-_ред.-от-22.09                                                                        4257
Федеральный закон № 39-ФЗ «О рынке ценных бумаг»                                                                                   1095
Федеральный закон от 02.12.90 N 395-I О банках и банковской деятельности                                                            845
Инструкция ЦБ РФ от 28.06.2017 N 180-И Об обязательных нормативах банков                                                            683
Федеральный закон от 30 декабря 2004 г N 214 ФЗ Об участии в долевом строительст                                                    592
Федеральный закон № 218-ФЗ "О публично-правовой компании «Фонд защиты прав граждан — участников долевого строительства»"            310
Положение Банка России № 590-П (06.07.2022)                                                                                         261
Постановление Правительства РФ от 11 ок

Модели для русского языка: bi-encoder `intfloat/multilingual-e5-small` (префиксы `query:` / `passage:`), реранкер `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (многоязычный MiniLM, дообученный на mMARCO).

In [12]:
ru_embedder = SentenceTransformer('intfloat/multilingual-e5-small', device=device)
ru_reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', device=device, max_length=512)

legal_index = Index(legal_chunks, 'ru', ru_embedder, ru_reranker,
                    cache=f'{CACHE}/legal_emb_{len(legal_chunks)}.npy')

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

c:\Users\Lenovo\miniconda3\envs\jupyter_env\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--intfloat--multilingual-e5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.28k [00:00<?, ?B/s]

Batches:   0%|          | 0/137 [00:00<?, ?it/s]

In [13]:
ask(legal_index, 'Что такое кредитная организация?', 'ru')

Вопрос: Что такое кредитная организация?

Ответ: Кредитная организация - юридическое лицо, которое для извлечения прибыли как основной цели своей деятельности на основании специального разрешения (лицензии) Центрального банка Российской Федерации (Банка России) имеет право осуществлять банковские операции, предусмотренные настоящим Федеральным законом. Кредитная организация образуется на основе любой формы собственности как хозяйственное общество.

Источники:
  [1] Федеральный закон от 02.12.90 N 395-I О банках и банковской деятельнос | 'N 201-ФЗ, от 08.08.2024 N 243-ФЗ, от 08.08.2024 N 260-ФЗ, от 08.08.2024 N 275-ФЗ, от 28.12.2024 N 522-ФЗ, от 28.12.2024 N 532-ФЗ, от 13.02.2025 N 9-ФЗ'
  [2] Федеральный закон от 02.12.90 N 395-I О банках и банковской деятельнос | 'Статья 11.1. Органы управления кредитной организации (в ред. Федерального закона от 29.07.2017 N 281-ФЗ)\nстраховыми или клиринговыми организациями, пр'
  [3] Федеральный закон от 02.12.90 N 395-I О банках и банковской деяте

In [14]:
ask(legal_index, 'Какие требования предъявляются к застройщику для привлечения средств дольщиков?', 'ru')

Вопрос: Какие требования предъявляются к застройщику для привлечения средств дольщиков?

Ответ: Требования, предусмотренные частью 1.1, не применяются, а соответствие этим требованиям не требуется для застройщиков, привлекающих средства дольщиков, размещаемых на эскроу счетах.

Источники:
  [1] Федеральный закон от 30 декабря 2004 г N 214 ФЗ Об участии в долевом с | 'Статья 3. Право на привлечение денежных средств участников долевого строительства для строительства (создания) многоквартирного дома и (или) иных объе'
  [2] Федеральный закон от 30 декабря 2004 г N 214 ФЗ Об участии в долевом с | 'Статья 3 дополнена частью 1.3 с 1 марта 2022\xa0г. - Федеральный закон от 30 декабря 2021\xa0г. N 476-ФЗ\nредакцию\nГАРАНТ:\nТребования, предусмотренные пункто'
  [3] Федеральный закон от 30 декабря 2004 г N 214 ФЗ Об участии в долевом с | 'Статья 3 дополнена пунктом 1.8 с 30 сентября 2018\xa0г. - Федеральный закон от 1 июля 2018\xa0г. N 175-ФЗ\nредакцию\nГАРАНТ:\nТребования, предусмотренные пун

In [15]:
# вопрос вне корпуса — ожидаем отказ
ask(legal_index, 'Какой срок исковой давности по трудовым спорам?', 'ru')

Вопрос: Какой срок исковой давности по трудовым спорам?

Ответ: Срок исковой давности для признания выпуска (дополнительного выпуска) эмиссионных ценных бумаг, решений, принятых эмитентом, Банком России и (или) регистрирующей организацией и связанных с осуществлением эмиссии ценных бумаг, недействительными, составляет три месяца с момента государственной регистрации отчета об итогах выпуска (дополнительного выпуска) эмиссионных ценных бумаг. Указанный в настоящем пункте срок исковой давности в случае его пропуска восстановлению не подлежит.

Источники:
  [1] Федеральный закон № 39-ФЗ «О рынке ценных бумаг» | 'Статья 26. Приостановление эмиссии ценных бумаг. Признание выпуска (дополнительного выпуска) эмиссионных ценных бумаг несостоявшимся или недействитель'
  [2] Указание-Банка-России-от-08.10.2018-N-4927-У-_ред.-от-22.09 | '0,05\n2.1.1.1.1.2со сроком исполнения обязательств, превышающим 30 календарных дней                         0,05\nна счетах физических лиц, имеющих уста'
  [3] Фе

### Оценка retrieval

In [16]:
legal_ret = eval_retrieval(legal_index, legal_qa, 'ru')
legal_ret

BM25:   0%|          | 0/25 [00:00<?, ?it/s]

Dense:   0%|          | 0/25 [00:00<?, ?it/s]

Hybrid (RRF):   0%|          | 0/25 [00:00<?, ?it/s]

Dense + rerank:   0%|          | 0/25 [00:00<?, ?it/s]

Hybrid + rerank:   0%|          | 0/25 [00:00<?, ?it/s]

,Hit@1,Hit@3,Hit@5,Hit@10,MRR@10,AnsRecall@5
BM25,0.76,0.88,0.96,0.96,0.838,0.770
Dense,0.88,0.92,0.92,0.96,0.904,0.892
Hybrid (RRF),0.84,0.88,0.92,0.96,0.873,0.861
Dense + rerank,0.96,0.96,0.96,0.96,0.960,0.924
Hybrid + rerank,0.96,0.96,0.96,0.96,0.960,0.924


### Оценка ответов
Сравниваем LLM без контекста и RAG с двумя вариантами поиска.

In [17]:
legal_preds = run_generation(legal_index, legal_qa['question'], 'ru', GEN_CONFIGS)
legal_gen = pd.DataFrame({n: score_answers(p, legal_qa['answer'], 'ru', ru_embedder) for n, p in legal_preds.items()}).T.round(3)
legal_gen

LLM без RAG:   0%|          | 0/25 [00:00<?, ?it/s]

RAG: dense:   0%|          | 0/25 [00:00<?, ?it/s]

RAG: hybrid + rerank:   0%|          | 0/25 [00:00<?, ?it/s]

,F1,SemSim,Refusal
LLM без RAG,0.245,0.916,0.00
RAG: dense,0.555,0.936,0.00
RAG: hybrid + rerank,0.553,0.923,0.04


In [18]:
# качество по типам вопросов (RAG: hybrid + rerank)
tok_ru = make_tokenizer('ru')
legal_qa['f1_rag'] = [token_f1(re.sub(r'\[\d+\]', '', p), g, tok_ru)
                      for p, g in zip(legal_preds['RAG: hybrid + rerank'], legal_qa['answer'])]
legal_qa['f1_plain'] = [token_f1(p, g, tok_ru) for p, g in zip(legal_preds['LLM без RAG'], legal_qa['answer'])]
legal_qa.groupby('question_type')[['f1_plain', 'f1_rag']].agg(['mean', 'count']).round(3)

f1_plain       f1_rag      
                  mean count   mean count
question_type                            
comparison       0.260     1  0.776     1
definition       0.243    13  0.740    13
factual          0.280     5  0.423     5
legal            0.209     5  0.263     5
process          0.264     1  0.000     1

In [19]:
for i in [0, 6, 11, 18]:
    print(f"Вопрос:   {legal_qa['question'][i]}\nЭталон:   {legal_qa['answer'][i]}\n")
    for n, p in legal_preds.items():
        print(f'{n}:\n  {p}\n')
    print('=' * 100)

Вопрос:   Что такое кредитная организация согласно закону о банках?
Эталон:   Кредитная организация - юридическое лицо, которое для извлечения прибыли как основной цели своей деятельности на основании специального разрешения (лицензии) Центрального банка Российской Федерации (Банка России) имеет право осуществлять банковские операции, предусмотренные Федеральным законом. Кредитная организация образуется на основе любой формы собственности как хозяйственное общество.

LLM без RAG:
  ['Кредитная организация, согласно законодательству Российской Федерации, - это юридическое лицо, которое осуществляет банковскую деятельность, включая предоставление кредитов, займа, депозитов и другие виды финансовых услуг. Это могут быть банки, микрофинансовые организации, кредитные кооперативы и другие организации, которые имеют лицензию на банковскую деятельность.', 'Банки с универсальной лицензией и базовой лицензией имеют некоторые различия в правах и обязанностях:\n\n1. Универсальная лицензия:\n- Позв

## Часть 2. TechQA-RAG-Eval (IBM, английский)
Корпус — 28 тыс. технотов (~105 МБ текста). Чтобы индексация не шла часами на CPU, берём все документы, на которые ссылаются вопросы (496 шт.), и `N_DISTRACTORS` случайных документов-«шумов». `N_DISTRACTORS = None` — весь корпус.

In [23]:
TQ = 'nvidia/TechQA-RAG-Eval'
tq_qa = pd.read_json(hf_hub_download(TQ, 'train.json', repo_type='dataset'))
tq_qa['gold_docs'] = tq_qa['contexts'].apply(lambda cs: {c['filename'] for c in cs})

zf = zipfile.ZipFile(hf_hub_download(TQ, 'corpus.zip', repo_type='dataset'))
names = {os.path.basename(n): n for n in zf.namelist() if n.endswith('.txt')}
gold = set().union(*tq_qa['gold_docs'])

N_DISTRACTORS = 500
others = sorted(set(names) - gold)
if N_DISTRACTORS is not None:
    others = random.Random(42).sample(others, N_DISTRACTORS)
tq_docs = {n: zf.read(names[n]).decode('utf-8', errors='replace') for n in sorted(gold) + others}

print(f'вопросов: {len(tq_qa)}, без ответа: {tq_qa.is_impossible.sum()}, документов в индексе: {len(tq_docs)}')
tq_qa[['question', 'answer', 'is_impossible']].head()

вопросов: 910, без ответа: 300, документов в индексе: 996


,question,answer,is_impossible
0,"User environment variables no longer getting picked up after upgrade to 4.1.1.1 or 4.1.1.2?\n\n\n\nHave you found that after upgrade to Streams 4.1.1.1 or 4.1.1.2, that environment variables set i...","To work around the issue, set environment variables that are needed by the application directly in the instance with: * \n \n * streamtool setproperty\n * -d <domain> -i <instance>\n --applic...",False
1,Netcool/Impact (all versions): How is the Exit() action function expected to work with User Defined Functions?\n\nNetcool/Impact (all versions)\n\nUsing the Exit() action function within a User De...,This is because the Exit() parser function in IPL is designed to exit the immediate scope. To carry the action outside of the User Defined Function to the Policy level one would have to set a vari...,False
2,Why are replies going to the DLQ with reason 2189 MQRC_CLUSTER_RESOLUTION_ERROR\n\nI have changed my cluster receiver to match the IP address used and now I am seeing replies in my DLQ with 2189 M...,-,True
3,How to configure SSL mutual authentication in IBM HTTP Server?\n\n\n\nWe are running IHS v7 and our application team has a specific webservice that they require SSL mutual authentication to be ena...,The following steps help guide you through the proper set up of SSL within the IBM HTTP Server: \n\n 1. Confirm that the Global Security Kit (GSKit) is installed and meets the minimum requirements...,False
4,Help with Action required for IIB H.E. V9 & WMB H.E. V8 for security vulnerabilities in Red Hat Linux (CVE-2018-6551 CVE-2018-6485)\n\n\n\nI need to understand details regarding Action required fo...,-,True


In [24]:
def tq_title(name, text):
    first = text.split('\n', 1)[0]
    return first.replace(' - United States', '')

tq_chunks = build_chunks(tq_docs, size=1200, overlap=200, title_fn=tq_title)
print(len(tq_chunks))
print(tq_chunks['text'].iloc[1])

chunking:   0%|          | 0/996 [00:00<?, ?it/s]

4969
Title: IBM PI19804: SQLCODE401 FROM DSNXOBFC FOR A SQL PROCEDURE WHEN USING INLIST IN IF STATEMENT
*
*                 predicate which references SQL variables.    *
****************************************************************
* PROBLEM DESCRIPTION: SQLCODE -401 or unexpected ABEND0C4     *
*                      or ABEND04E may occur for a CREATE      *
*                      PROCEDURE/FUNCTION statement that       *
*                      contains control statement with an      *
*                      IN-LIST predicate which references SQL  *
*                      variables.                              *
*                                                              *
****************************************************************
* RECOMMENDATION:                                              *
****************************************************************
The following examples help to illustrate the problem.
Example#1. statement which issued SQLCODE -401 incorrectly.

Для английского: bi-encoder `intfloat/e5-small-v2`, реранкер `cross-encoder/ms-marco-MiniLM-L-6-v2` (те же, что на уроке).

In [25]:
en_embedder = SentenceTransformer('intfloat/e5-small-v2', device=device)
en_embedder.max_seq_length = 256  # чанк ~1200 символов ≈ 300 токенов; короче контекст — быстрее
if device == 'cuda':
    en_embedder.half()
en_reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device, max_length=512)

tq_index = Index(tq_chunks, 'en', en_embedder, en_reranker,
                 cache=f'{CACHE}/techqa_emb_{len(tq_chunks)}.npy')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/78 [00:00<?, ?it/s]

In [26]:
ask(tq_index, tq_qa['question'][0], 'en')
print('\nЭталон:', tq_qa['answer'][0])

Вопрос: User environment variables no longer getting picked up after upgrade to 4.1.1.1 or 4.1.1.2?



Have you found that after upgrade to Streams 4.1.1.1 or 4.1.1.2, that environment variables set in your .bashrc are no longer being set? For example ODBCINI is not set for the database toolkit and you get

     An SQL operation failed. The SQL state is 08003, the SQL code
     is 0 and the SQL message is [unixODBC][Driver
     Manager]Connnection does not exist.


Ответ: I don't know.

Источники:
  [1] swg21996508.txt | 'Title: IBM STREAMS 4.1.1.1 and 4.1.1.2  JOBS DO NOT INHERIT THE ENVIRONMENT VARIABLES SET IN .BASHRC, WHEN STREAMS IS RUN AS A SYSTEM SERVICE - United'
  [2] swg21996508.txt | 'set environment variables that are needed by the application directly in the instance with:  *\n* streamtool setproperty\n* -d <domain> -i <instance>\n--'
  [3] swg21982857.txt | 'Title: IBM User receiving Notice Sender Execution Failed Alert after upgrading to Atlas 6.0.3.3 - United States\nTe

### Оценка retrieval
Только на вопросах, у которых есть ответ. `N_RET_EVAL` вопросов — реранкинг 50 кандидатов на каждый вопрос не бесплатный.

In [27]:
N_RET_EVAL = 300
tq_ans = tq_qa[~tq_qa['is_impossible']].sample(N_RET_EVAL, random_state=42).reset_index(drop=True)
tq_ret = eval_retrieval(tq_index, tq_ans, 'en')
tq_ret

BM25:   0%|          | 0/300 [00:00<?, ?it/s]

Dense:   0%|          | 0/300 [00:00<?, ?it/s]

Hybrid (RRF):   0%|          | 0/300 [00:00<?, ?it/s]

Dense + rerank:   0%|          | 0/300 [00:00<?, ?it/s]

Hybrid + rerank:   0%|          | 0/300 [00:00<?, ?it/s]

,Hit@1,Hit@3,Hit@5,Hit@10,MRR@10,AnsRecall@5
BM25,0.700,0.797,0.840,0.890,0.760,0.864
Dense,0.727,0.830,0.860,0.887,0.782,0.877
Hybrid (RRF),0.767,0.857,0.880,0.920,0.818,0.897
Dense + rerank,0.720,0.810,0.870,0.900,0.780,0.871
Hybrid + rerank,0.723,0.820,0.867,0.913,0.784,0.869


### Оценка ответов
Отдельно смотрим на вопросы с ответом (качество и ложные отказы) и без ответа (доля правильных отказов — модель не должна выдумывать).

In [ ]:
N_GEN_ANS, N_GEN_IMP = 40, 20
gen_ans = tq_ans.head(N_GEN_ANS)
gen_imp = tq_qa[tq_qa['is_impossible']].sample(N_GEN_IMP, random_state=42).reset_index(drop=True)

tq_preds_ans = run_generation(tq_index, gen_ans['question'], 'en', GEN_CONFIGS)
tq_preds_imp = run_generation(tq_index, gen_imp['question'], 'en', GEN_CONFIGS)

LLM без RAG:   0%|          | 0/10 [00:00<?, ?it/s]

RAG: dense:   0%|          | 0/10 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
refusal = PROMPTS['en']['refusal'].lower()
tq_gen = pd.DataFrame({
    n: {**score_answers(tq_preds_ans[n], gen_ans['answer'], 'en', en_embedder),
        'Отказ на вопросах без ответа': np.mean([refusal in p.lower() for p in tq_preds_imp[n]])}
    for n in GEN_CONFIGS}).T.round(3)
tq_gen.rename(columns={'Refusal': 'Ложный отказ'})

In [ ]:
for i in range(3):
    print(f"Question: {gen_ans['question'][i][:400]}\n\nЭталон: {gen_ans['answer'][i]}\n")
    for n, p in tq_preds_ans.items():
        print(f'{n}:\n  {p}\n')
    print('=' * 100)

## Итог

In [ ]:
pd.concat({'Legal RU': legal_ret, 'TechQA EN': tq_ret}, axis=1)

In [ ]:
pd.concat({'Legal RU': legal_gen, 'TechQA EN': tq_gen}, axis=1)